# P&ID Medallion Pipeline — Concepts Walkthrough (Bronze → Silver)

This notebook illustrates, end to end, what we have built so far: a **Bronze**
(raw, immutable ingestion) → **Silver** (parse + topology reconstruction) pipeline
for P&ID interoperability exports (DEXPI/Proteus and INGR ISO-15926 PostProc),
on local Spark + Delta Lake.

It demonstrates the key concepts:

- **Bronze** stores the source XML *as-is* — content hash, format detection, project
  code and drawing revision captured, dedup on exact bytes.
- **Silver** *re-houses* the validated `pidtool`/`bppidsys` reconstruction (the
  "crown jewel") — recovering inline valves the raw graph lacks — and emits typed
  tables: components, segments, connections, equipment.
- The **oracle firewall**: the source turnover assignment is carried as *quarantined*
  lineage, never computed on.
- The **`flow_sense`** four-state directional overlay and the **`derived`** provenance
  flag on every reified connection.
- **Format parity**: DEXPI and PostProc flow through one code path into one schema.
- A real-data finding: **`seg_tag` is not unique** (the CDC anchor-collision risk).
- **Stage D**: a declarative expectation suite writes the **`silver_quality`**
  punch list; only two structural invariants hard-fail (fail for bugs, not data).

> **Run order matters.** After any kernel restart, run the cells top-to-bottom.
> Cell 1 *must* be first — it forces the venv's Spark 3.5.1 and blocks the system
> Spark 4 at `/opt/spark`.

## 0. Environment & pinned session

Two things bite on local WSL and are handled here:

1. **Which Spark.** A system `SPARK_HOME=/opt/spark` (Spark 4) shadows the venv's
   Spark 3.5.1 and breaks Delta (`DeltaCatalog` not found). Cell 1 strips it
   *before* `pyspark` is ever imported.
2. **One catalog, one warehouse.** The Hive metastore (`metastore_db/`) holds
   *names → locations*; the warehouse (`spark-warehouse/`) holds the *data*. We
   **pin both** to fixed paths so every session sees the same tables (embedded
   Derby is single-session — don't also run a `!python -m ...` subprocess while
   this notebook's session is live).

In [ ]:
# --- CELL 1 — must run FIRST (before any `import pyspark`) ---
import os, sys
os.environ.pop("SPARK_HOME", None)                       # ignore system /opt/spark (Spark 4)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "Restart the kernel and run THIS cell first."

from pathlib import Path

# repo_root = the ProjectData repo root (the folder containing bronze/ and silver/)
repo_root = Path.cwd()
while not (repo_root / "bronze").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "bronze").is_dir(), f"Open this notebook inside the ProjectData repo (cwd={Path.cwd()})"

# --- the variables for this walkthrough ---
source_sample_dir = repo_root / "sample_data"                 # committed synthetic fixtures
source_dir        = repo_root / "data/exports/projectA"       # real Project A (DEXPI)
source_dir_B      = repo_root / "data/exports/projectB"       # real Project B (PostProc)
bronze_table      = "bronze.pid_documents"                    # NAMED Bronze table (metastore)
spark_warehouse   = repo_root / "spark-warehouse"             # managed-table data
metastore_db      = repo_root / "metastore_db"                # Hive/Derby catalog

# --- metastore hygiene (run BEFORE the session, Cell 2) ---
# Embedded Derby allows ONE connection. A leftover lock from a crashed or still-open
# kernel makes a new session fail with "Unable to instantiate SessionHiveMetaStoreClient".
# Clear stale locks here; flip RESET=True for a guaranteed clean slate — the metastore
# + warehouse are a throwaway PoC store, everything is rebuilt from the XML below.
import shutil
RESET = False        # set True, re-run this cell, then run the notebook top-to-bottom
if RESET:
    shutil.rmtree(metastore_db, ignore_errors=True)
    shutil.rmtree(spark_warehouse, ignore_errors=True)
for _lck in ("db.lck", "dbex.lck"):            # release a stale Derby lock (safe if unheld)
    try:
        (metastore_db / _lck).unlink(missing_ok=True)
    except Exception:
        pass
(repo_root / "derby.log").unlink(missing_ok=True)

print("repo_root       :", repo_root)
print("bronze_table    :", bronze_table)
print("spark_warehouse :", spark_warehouse)
print("metastore_db    :", metastore_db)
print("RESET           :", RESET)

In [ ]:
# --- CELL 2 — build ONE pinned Delta+Hive session (venv Spark 3.5.1) ---
from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{spark_warehouse}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={metastore_db};create=true",
})
import pyspark
from pyspark.sql import functions as F
print("pyspark :", pyspark.__file__)   # expect .../.venv/...  NOT /opt/spark
print("Spark   :", spark.version)      # expect 3.5.1
assert "/opt/spark" not in pyspark.__file__, "Still on system Spark 4 — restart kernel, run Cell 1 first."
assert spark.version.startswith("3.5"), f"Expected Spark 3.5.x, got {spark.version}"
print("OK — venv Spark 3.5.1, Delta + Hive ready.")

In [ ]:
# --- CELL 3 (optional) — targeted rebuild (needs a WORKING metastore) ---
# Drops the named Bronze + Silver tables and their warehouse dirs so a re-run
# starts fresh (avoids DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION). If the
# metastore itself is wedged ("Unable to instantiate SessionHiveMetaStoreClient"),
# this cell can't help — use Cell 1's RESET=True instead (a filesystem wipe that
# runs before the session). Everything is rebuilt from the source XML below.
import shutil
_wipe = {
    "bronze": ["pid_documents"],
    "silver": ["silver_components", "silver_segments", "silver_connections",
               "silver_equipment", "silver_quality"],
}
for db, tables in _wipe.items():
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    for t in tables:
        spark.sql(f"DROP TABLE IF EXISTS {db}.{t}")
        shutil.rmtree(spark_warehouse / f"{db}.db" / t, ignore_errors=True)
print("clean slate ready")

## 1. Bronze — raw, immutable ingestion

Bronze lands each source file **verbatim**, one row per distinct byte-version, with:
`content` (raw bytes), a self-describing `content_hash` (`sha256:…`), the detected
`source_format` (DEXPI vs POSTPROC), the EPC `document_number` and derived
`project_code`, and the current `drawing_revision` / `drawing_revision_date`.
It **never interprets** the network model — that's Silver's job.

We ingest into the **named** Bronze table `bronze.pid_documents` in the pinned
metastore, and every stage below reads and writes named tables in that one
catalog — so the whole notebook is consistent (no path-based side door).

In [ ]:
# --- pick sources: prefer the real exports, fall back to the committed samples ---
def xmls(d): return sorted(Path(d).glob("*.xml")) if Path(d).is_dir() else []
sources = [d for d in (source_dir, source_dir_B) if xmls(d)]
if not sources:
    sources = [source_sample_dir]
for d in sources:
    print(f"{len(xmls(d)):3d} xml  in  {d}")

In [ ]:
# --- ingest each source folder into the SAME named Bronze table ---
from bronze.notebook import ingest_folder
for d in sources:
    summary = ingest_folder(spark, source_dir=str(d), table_name=bronze_table)
    print(d.name, "->", summary)

In [ ]:
# --- inspect Bronze: both formats, lineage columns, self-describing hash ---
bronze = spark.table(bronze_table)
print("Bronze rows:", bronze.count())
bronze.groupBy("source_format").count().show()
bronze.select("document_number", "drawing_revision", "drawing_revision_date",
              "project_code", "content_hash").show(6, False)

**Dedup on exact bytes.** Re-ingesting the same files lands *nothing* new —
Bronze versions files by `content_hash`, so identical bytes are skipped
(`rows_skipped_already_present`).

In [ ]:
# re-ingest the first folder — expect rows_inserted: 0
print(ingest_folder(spark, source_dir=str(sources[0]), table_name=bronze_table))

## 2. Silver — parse + topology reconstruction

Silver reads Bronze, picks the adapter from `source_format`, builds the DOM from
the raw bytes, and runs the **validated reconstruction** (vendored under
`silver/_recon/`, re-housed not re-derived). It emits four typed Delta tables and
carries the source turnover assignment as **quarantined** lineage.

We run it **in-session** (same notebook session) reading the named Bronze table —
so the Silver tables land in this session's pinned catalog and are queryable by
name.

In [ ]:
# --- run Silver Stage A+B in-session ---
from silver.notebook import reconstruct
counts = reconstruct(spark, bronze_table=bronze_table, silver_schema="silver")
print(counts)

In [ ]:
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    print(f"{t:22s} {spark.table('silver.' + t).count():6d} rows")

## 3. The concepts, illustrated in the data

### 3a. The crown jewel — inline valves recovered

The raw `<Connection>` records wire only each segment's two endpoints; inline valves
are missing. The reconstruction repairs the topology and re-inserts them. Here they
appear as real components flagged `is_valve` — in **both** formats.

In [ ]:
spark.table("silver.silver_components") \
     .groupBy("source_format", "is_valve").count() \
     .orderBy("source_format", "is_valve").show()

### 3b. The oracle firewall

`src_turnover` / `src_subsystem` (the source commissioning assignment) is **carried**
on the segment row — but it sits on its own columns and **nothing computes on it**.
It is the validation *answer key*, quarantined so the ~97% agreement stays honest.

In [ ]:
spark.table("silver.silver_segments") \
     .select("seg_tag", "fluid", "piping_materials_class",
             "src_turnover", "src_subsystem", "project_code").show(6, False)

### 3c. `flow_sense` — the four-state directional overlay

Direction is a *separate overlay* on the undirected connection, and it has four
states — `none` and `both` are real and a boolean couldn't hold them. Both formats
produce all four.

In [ ]:
spark.table("silver.silver_connections") \
     .groupBy("source_format", "flow_sense").count() \
     .orderBy("source_format", "flow_sense").show()

### 3d. `derived` — Source (stated) vs Derived (reconstructed) edges

Every reified connection carries provenance: `derived=false` where the edge was
stated in a source `<Connection>`, `derived=true` where the reconstruction inferred
it. This is what keeps the semantic layer from asserting inferred topology as fact.

In [ ]:
spark.table("silver.silver_connections").groupBy("source_format", "derived").count().show()

### 3e. Format parity — two standards, one schema

DEXPI and PostProc coexist in the same tables with identical columns — the
interoperability promise made concrete.

In [ ]:
spark.table("silver.silver_segments").groupBy("source_format").count().show()

### 3f. Real-data finding — `seg_tag` is not unique

Distinct `segment_id`s can compose to the **same** business `seg_tag`. So the
composed tag cannot stand alone as the CDC segment anchor — it needs a
disambiguator, and the quality gate owes an *anchor-collision* flag. (This is why
we recorded it in the spec's §3.5.)

In [ ]:
(spark.table("silver.silver_segments")
   .groupBy("seg_tag").count().filter("count > 1")
   .orderBy(F.desc("count")).show(10, False))

### 3f-diagnostics — how `(Drawing, Piping Segment)` identity actually occurs

Three probes over the **real Project-B** data to size the anchor-collision problem
and test the "position in the Piping System" disambiguator (§3.5). Everything is
scoped by `source_format` (set `PROJECT_FMT`) and grouped by `content_hash`, so one
*version* of a drawing is one Bronze content hash — correct whether one or several
revisions are loaded. (The committed sample fixtures are minimal — run this against
the real Project-B exports to see genuine collisions.)

**(1) Is `(drawing, seg_tag)` unique?** The distribution of anchor multiplicity.

In [ ]:
# --- how (Drawing, Piping Segment) identity occurs — Project B diagnostics ---
PROJECT_FMT = "POSTPROC"     # Project B (ISO-15926 PostProc); use "DEXPI" for Project A

spark.sql(f'''
WITH anchor AS (
  SELECT drawing_number, content_hash AS version, seg_tag,
         COUNT(DISTINCT segment_id) AS n_segments
  FROM silver.silver_segments
  WHERE source_format = '{PROJECT_FMT}'
  GROUP BY drawing_number, content_hash, seg_tag
)
SELECT CASE
         WHEN seg_tag IS NULL OR trim(seg_tag) = '' THEN '0 no seg_tag (degenerate)'
         WHEN n_segments = 1                         THEN '1 unique'
         ELSE format_string('%d COLLISION', n_segments)
       END             AS anchor_multiplicity,
       COUNT(*)        AS num_anchors,
       SUM(n_segments) AS num_segments
FROM anchor GROUP BY 1 ORDER BY anchor_multiplicity''').show(20, False)

**(2) Where it collides, what would separate the segments cheaply?** Whether the
Piping-System tag / subline / attributes already differ among the colliding
segments (if `distinct_pns = n_segments`, `pns_tag` alone disambiguates; if every
`distinct_*` is 1, only connectivity/geometry can).

In [ ]:
spark.sql(f'''
WITH coll AS (
  SELECT drawing_number, content_hash, seg_tag
  FROM silver.silver_segments
  WHERE source_format='{PROJECT_FMT}' AND seg_tag IS NOT NULL AND trim(seg_tag) <> ''
  GROUP BY drawing_number, content_hash, seg_tag
  HAVING COUNT(DISTINCT segment_id) > 1
)
SELECT s.drawing_number, s.seg_tag,
       COUNT(DISTINCT s.segment_id)                                        AS n_segments,
       COUNT(DISTINCT s.pns_tag)                                           AS distinct_pns,
       COUNT(DISTINCT s.subline_tag)                                       AS distinct_subline,
       COUNT(DISTINCT concat_ws('|', s.diameter, s.piping_materials_class,
                                     s.insul_purpose))                     AS distinct_attrs
FROM silver.silver_segments s
JOIN coll c ON s.drawing_number=c.drawing_number AND s.content_hash=c.content_hash
           AND s.seg_tag=c.seg_tag
WHERE s.source_format='{PROJECT_FMT}'
GROUP BY s.drawing_number, s.seg_tag
ORDER BY n_segments DESC''').show(30, False)

**(3) The connection-order test** — do the colliding segments sit between
*different* lines? The stable "endpoint signature": for each colliding segment, the
set of neighbouring `seg_tag`s (via its components' connections). Distinct neighbour
sets ⇒ the position-in-the-Piping-System disambiguator works (and is UID-free);
identical sets ⇒ a residual case needing geometry / OPC-nozzle endpoints.

In [ ]:
spark.sql(f'''
WITH comp AS (
  SELECT content_hash, component_id, segment_id
  FROM silver.silver_components
  WHERE source_format='{PROJECT_FMT}' AND segment_id IS NOT NULL
),
edges AS (
  SELECT content_hash, from_id AS a, to_id AS b FROM silver.silver_connections WHERE source_format='{PROJECT_FMT}'
  UNION ALL
  SELECT content_hash, to_id AS a, from_id AS b FROM silver.silver_connections WHERE source_format='{PROJECT_FMT}'
),
seg_adj AS (
  SELECT DISTINCT e.content_hash, ca.segment_id AS seg, sb.seg_tag AS nbr_tag
  FROM edges e
  JOIN comp ca ON ca.content_hash=e.content_hash AND ca.component_id=e.a
  JOIN comp cb ON cb.content_hash=e.content_hash AND cb.component_id=e.b
  JOIN silver.silver_segments sb
       ON sb.content_hash=e.content_hash AND sb.segment_id=cb.segment_id
  WHERE ca.segment_id <> cb.segment_id
),
coll AS (
  SELECT drawing_number, content_hash, seg_tag
  FROM silver.silver_segments
  WHERE source_format='{PROJECT_FMT}' AND seg_tag IS NOT NULL AND trim(seg_tag) <> ''
  GROUP BY drawing_number, content_hash, seg_tag
  HAVING COUNT(DISTINCT segment_id) > 1
)
SELECT s.drawing_number, s.seg_tag, s.segment_id, s.pns_tag,
       sort_array(collect_set(a.nbr_tag)) AS neighbour_line_tags,
       COUNT(a.nbr_tag)                   AS inter_segment_degree
FROM silver.silver_segments s
JOIN coll c ON s.drawing_number=c.drawing_number AND s.content_hash=c.content_hash
           AND s.seg_tag=c.seg_tag
LEFT JOIN seg_adj a ON a.content_hash=s.content_hash AND a.seg=s.segment_id
WHERE s.source_format='{PROJECT_FMT}'
GROUP BY s.drawing_number, s.seg_tag, s.segment_id, s.pns_tag
ORDER BY s.drawing_number, s.seg_tag, s.segment_id''').show(40, False)

**Resolution — the grain is the LINE, not the physical segment.** Probe (1)
on the real exports shows ~78 % of tags are shared by 2+ `PipingNetworkSegment`s,
and probe (2) shows the colliding pieces almost always agree on every attribute
(`distinct_attrs = 1`) — they are *not* different objects that happen to share a
tag, they are **one line drawn as many pieces**. Probe (3) confirms the internal
pieces have identical neighbour sets, so a within-line position key cannot separate
them (nor should it — they are the same line). The design conclusion, now in
`silver_spec` §3.5: don't disambiguate the pieces, **aggregate** them. `cdc.aggregate_lines`
collapses `(drawing, seg_tag)` to one line object (attributes as distinct-value
*sets*, routing as the neighbour-**line** set); a pure re-split is inert, a real
change surfaces, and a genuine within-line disagreement (`distinct_attrs > 1`) is a
spec break Stage D quarantines up front (`line_attr_inconsistent`) — never averaged
away. Un-composable tags (probe would show them as *degenerate* / no numeric core)
are excluded via the shared `is_uncomposable_seg_tag` predicate.

## 3g. Stage D — the data-quality punch list

Stage D promotes the specs' advisory flags to a **declarative expectation suite**
(rules-as-data, `silver/quality_suite.py`) and writes `silver_quality` — the
per-drawing / per-project **punch list** a pre-commissioning engineer fixes at
source *before* systemization runs (segments missing fluid / piping-class /
diameter, tags that break the naming convention, un-composable seg_tags and
within-line attribute inconsistencies held out of line CDC, the `seg_tag`
line-multiplicity report, prefix-integrity, orphans). The gate is
**observe-and-record**: everything flags
and flows. Only two *structural invariants* — an **oracle leak** (§5) or an
**unflagged `derived` edge** (§4) — hard-fail, because those are pipeline bugs,
not dirty data.

It also denormalises a `quality_gate` enum (`clean`/`flagged`/`quarantined`) back
onto every object row, so a cautious consumer can filter without joining the
ledger.

In [ ]:
# --- run Stage D in-session; it reads the four Silver tables ---
from silver.notebook import quality
# refdata_path lights up the reference-backed checks (unknown fluid/unit, naming);
# without it those skip cleanly. Point it at the project's Reference_Data.xlsx:
refdata_path = repo_root / "Reference_Data.xlsx"
summary = quality(spark, refdata_path=str(refdata_path) if refdata_path.exists() else None)
import json; print(json.dumps(summary, indent=2, default=str))

**The punch list** — one row per flag occurrence, ordered worst-first. This
is the artefact the engineer works from.

In [ ]:
from pyspark.sql import functions as F
sev_rank = F.when(F.col("severity") == "error", 0).when(F.col("severity") == "warn", 1).otherwise(2)
(spark.table("silver.silver_quality")
   .withColumn("_r", sev_rank)
   .orderBy("_r", "flag")
   .select("severity", "gate", "object_kind", "flag", "drawing_number", "detail")
   .show(40, False))

**Punch-list rollup by flag** — where the data gaps concentrate.

In [ ]:
(spark.table("silver.silver_quality")
   .groupBy("flag", "severity", "gate").count()
   .orderBy(F.desc("count")).show(30, False))

**The gate rollup on the objects themselves** — `flagged` rows still flow to
Gold and the rules; a strict consumer can exclude `quarantined` without a join.

In [ ]:
for t in ["silver_segments", "silver_components"]:
    print(t)
    spark.table("silver." + t).groupBy("quality_gate").count().orderBy("quality_gate").show()

## 3h. Lineage trace — one attribute, Bronze bytes → Silver column

The whole point of carrying `bronze_id` / `content_hash` on every Silver row (§4)
is that any value traces back to the exact source bytes it came from. Here we
follow **insulation** end to end: the Silver `insul_purpose` column, the raw
`InsulPurpose` attribute pulled straight out of the Bronze XML, and the `seg_tag`
suffix (e.g. `-H`) are the *same source fact reached three ways*. Reading it from
the segment's own `<GenericAttributes>` block mirrors `pidsys.master_data.ga()`
exactly. The identical three-hop walk traces fluid, diameter, piping class, or the
quarantined oracle columns — insulation isn't special.

In [ ]:
# --- Lineage trace: Insulation from Bronze bytes -> Silver columns ---
import xml.etree.ElementTree as ET

seg = spark.table("silver.silver_segments")

# 1) the Silver insulation columns + the lineage keys that trace each row to source
(seg.select("segment_id", "seg_tag", "insul_purpose", "insul_type", "insul_thick",
            "bronze_id", "content_hash", "drawing_number")
    .where("insul_purpose is not null")
    .show(8, False))

# 2) pull InsulPurpose straight out of the raw Bronze XML for one segment and compare.
bronze = spark.table(bronze_table)

row = (seg.where("insul_purpose is not null")
          .join(bronze.select("bronze_id", "content"), "bronze_id")
          .select("segment_id", "insul_purpose", "insul_type", "insul_thick", "content")
          .head())

def _ln(el):                                   # strip XML namespace
    return el.tag.split("}")[-1]

def insul_from_bytes(content, seg_id):
    # mirror pidsys.master_data.ga(): the segment's OWN <GenericAttributes> block
    root = ET.fromstring(bytes(content))
    for el in root.iter():
        if _ln(el) == "PipingNetworkSegment" and el.get("ID") == seg_id:
            return {g.get("Name"): g.get("Value")
                    for gas in el if _ln(gas) == "GenericAttributes"
                    for g in gas if _ln(g) == "GenericAttribute"
                    and (g.get("Name") or "").startswith("Insul")}
    return {}

if row is None:
    print("no segment with a non-null insul_purpose yet — run Silver Stage A+B first")
else:
    tag = seg.where(seg.segment_id == row.segment_id).head().seg_tag
    print("segment_id :", row.segment_id)
    print("seg_tag    :", tag, "  (last token = insulation purpose)")
    print("SILVER cols:", dict(insul_purpose=row.insul_purpose,
                               insul_type=row.insul_type, insul_thick=row.insul_thick))
    print("BRONZE XML :", insul_from_bytes(row.content, row.segment_id))
    # to trace a SPECIFIC flagged segment: replace the filter in `row` with
    #   .where("segment_id = '<the id from silver_quality.object_id>'")

> If `insul_purpose` comes back all-null in Silver while the `seg_tag` still
> shows a `-H`/`-N` suffix, that mismatch *is* the finding — the value reached the
> composed tag but the column extraction missed it (Bronze→Silver drift), which is
> exactly what this trace is built to catch.

## 3i. Stage C — joining the P&IDs (off-page connectors)

Stage B reconstructs each drawing on its own; **Stage C** joins them into one
plant by matching **off-page connectors** (OPCs) across sheets — by `OPCTag` for
PostProc, by GUID for DEXPI (`bppidsys.offpage.match_pairs`, re-housed). Each
matched pair becomes one undirected, always-`derived` **`OffPage`** edge in
`silver_connections` that spans two drawings; an OPC whose mate isn't in the
loaded set is an **open boundary** — the system continues off-sheet — recorded as
an `opc_open_boundary` flag, never dropped. Run it after reconstruct().

In [ ]:
# --- run Stage C in-session (harvest OPCs per sheet -> match across sheets) ---
from silver.notebook import assemble
import json
stats = assemble(spark, bronze_table=bronze_table, silver_schema="silver")
print(json.dumps(stats, indent=2, default=str))

**The cross-document edges** — one row per stitched OPC pair, `derived=true`,
joining two drawings into one connectivity graph.

In [ ]:
off = spark.table("silver.silver_connections").where("conn_type = 'OffPage'")
print("OffPage edges:", off.count())
off.select("connection_id", "from_id", "to_id", "derived", "flow_sense").show(20, False)

**Open boundaries** — OPCs with no mate in the loaded set. Not errors: the
system continues onto a sheet that wasn't loaded. Load more sheets and these
resolve into `OffPage` edges.

In [ ]:
(spark.table("silver.silver_quality").where("flag = 'opc_open_boundary'")
   .select("object_id", "drawing_number", "detail").show(20, False))

## 3j. Stage E — change data capture, a two-revision narrative

SmartPlant re-exports the **whole** drawing XML for one symbol move, and re-mints
an element's UID when it is deleted and recreated — so a file hash (or the UID)
marks everything changed. **Stage E** answers the real questions with an
*anchor-match* identity that survives delete+recreate (equipment tag / **line**
`(drawing, seg_tag)` / `(line, class)` bucket) and three separated hashes: `anchor_hash`
(no UID), `content_hash_eng` (engineering attrs + neighbour **anchor** sets — the
Modify trigger), and `content_hash_audit` (adds UID + the quarantined oracle, so a
recreate is *visible* but stays *inert* for engineering CDC). Piping versions at
**line** grain — one line is drawn as many physical `PipingNetworkSegment` pieces
(on real Project-B data ~78 % of tags are shared), so `aggregate_lines` collapses
the pieces of `(drawing, seg_tag)` into one line object (attributes as distinct-value
*sets*, routing as the neighbour-**line** set) before diffing; a pure re-split is
then inert. It diffs, per drawing, the two most recent Bronze versions and writes
New/Modified/Deleted — the interval open/close events Gold consumes.

To make that concrete we run a **real EPC event**: a Project-B Unit-22 (Steam &
BFW) set is **issued at Rev C for HAZOP**, then **re-issued at Rev D for design**.
Between the two, engineering changed some lines / valves / instruments / equipment
— and *every element UID is re-minted*. We ingest both revisions as two Bronze
versions and let Stage E find the real change. (Runs on its **own** `silver_cdc_demo`
schema so the main walkthrough above is untouched.)

In [ ]:
# --- generate the two-revision narrative (4 synthetic PostProc XMLs) ---
from silver.demo_cdc import write_narrative, D1, D2
paths = write_narrative(str(repo_root / "_cdc_demo"))
print("Rev C:", paths["rev1_C"]); print("Rev D:", paths["rev2_D"])

In [ ]:
# --- run the mini medallion on an ISOLATED demo schema ---
import shutil, json
from bronze.notebook import ingest_folder
from silver.notebook import reconstruct, changes

demo_bronze, demo_schema = "bronze.cdc_demo", "silver_cdc_demo"
# clean any prior demo run (tables + warehouse dirs)
spark.sql(f"DROP TABLE IF EXISTS {demo_bronze}")
shutil.rmtree(spark_warehouse / "bronze.db" / "cdc_demo", ignore_errors=True)
for t in ["silver_components","silver_segments","silver_connections","silver_equipment","silver_cdc"]:
    spark.sql(f"DROP TABLE IF EXISTS {demo_schema}.{t}")
    shutil.rmtree(spark_warehouse / f"{demo_schema}.db" / t, ignore_errors=True)

# Rev C issued -> ingest + reconstruct (the plant as HAZOP saw it)
ingest_folder(spark, source_dir=paths["rev1_C"], table_name=demo_bronze)
reconstruct(spark, bronze_table=demo_bronze, silver_schema=demo_schema)
# Rev D re-issued -> append the second version, reconstruct again
ingest_folder(spark, source_dir=paths["rev2_D"], table_name=demo_bronze)
reconstruct(spark, bronze_table=demo_bronze, silver_schema=demo_schema)

# Stage E: the change report
summary = changes(spark, bronze_table=demo_bronze, silver_schema=demo_schema)
print(json.dumps(summary, indent=2, default=str))   # expect 2 drawings, ~15 deltas

**The change report** — grouped, then in detail. Every UID changed, yet only
the real engineering changes surface (`change_type` is the Gold interval event).

In [ ]:
cdc = spark.table(f"{demo_schema}.silver_cdc")
print("total deltas:", cdc.count())
cdc.groupBy("grain", "change_type").count().orderBy("grain", "change_type").show()
(cdc.select("grain", "change_type", "drawing_number", "anchor", "detail")
    .orderBy("grain", "change_type").show(60, False))

**Churn is invisible.** Every one of the ~30 elements was re-exported with a
new UID, but the re-drawn-yet-unchanged objects (line `WBF-2215103`, pump
`P-2201A`, both lines on Drawing 0016, drum `V-2202`) produce **no delta**. The
whole of Drawing 0016 was re-issued yet only its one new PSV vent line is flagged —
no engineer has to eyeball a re-issued sheet to find what moved.

Each delta is a **Gold** interval event: *New* opens `[validFrom, ∞)`, *Deleted*
closes the prior interval, *Modified* closes the old and opens the new — giving a
defensible "current truth" *and* a Rev-C-vs-Rev-D timeline, and scoping the
change-driven work (MTO delta, the new `PSV-2201` ITR, the drum-nozzle interface,
the redlined check valve) to exactly what changed. Full manifest by discipline:
`narrative_project_b/NARRATIVE.md`.

## 4. Recap

**Built (Phase-1 + Stages C, D, E):** Bronze (raw, immutable, dedup,
format-tagged) → Silver (parse + reconstruction, four typed tables) → **Stage C
assembly** (`OffPage` edges + open boundaries) → **Stage D quality gate**
(`silver_quality` punch list + `quality_gate` rollup) → **Stage E CDC**
(`silver_cdc` object-grain deltas, delete+recreate-safe), validated on real
Project A **and** Project B. Silver is complete.

**Concepts shown:** store-as-is + content hash; format detection; the reconstruction
recovering inline valves; the oracle firewall; the `flow_sense` enum and `derived`
provenance; format parity; the `seg_tag` anchor-collision; and the Stage-D
punch list with its fail-for-bugs-not-data gate policy.

**Runtime lessons baked in:** force the venv's Spark 3.5.1 (Cell 1); pin the metastore
+ warehouse; run everything **in-session** against the named tables in that one
metastore — no path-based side door, no `!python -m …` subprocess against a live Derby.

**Next:** the **Gold layer** — bi-temporal `validFrom`/`validTo` intervals over
Stage E's deltas, then the RDF/IDO projection and the Jena rule packages
(systemization, Test Packages). Stage D's reference-backed checks light up as soon
as a project `Reference_Data.xlsx` — with `Naming` and `Insulation` sheets — is
supplied.